# Exercise 1 · Simple RAG — DEMO & SOLUTIONS
**Lecturer notebook — run cell by cell while sharing the screen.**

```
question ──► ① search ──► ② chunks become context ──► ③ LLM call ──► answer with sources
```
Every step has its own cell, so the class sees exactly what comes back.

## 0 · Setup
Run the three cells below. The last one prints a greeting from the model.

In [1]:
!pip install -q unique-sdk


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [13]:
import json
import re
import unique_sdk

# --- API key -------------------------------------------------------------------
# COURSE KEY: only for this course, revoked after Module 1 (21 Nov 2026).
# Never put real keys into a notebook — in production, keys live in a secret store.
COURSE_API_KEY = "ukey_c56fQfqC31AulM6jru-rb33KYLTkip2dloEJ5eiUkOs"

try:  # the clean way: Colab Secrets (key icon in the left sidebar), if you set one
    from google.colab import userdata
    API_KEY = userdata.get("UNIQUE_API_KEY") or COURSE_API_KEY
except Exception:
    API_KEY = COURSE_API_KEY

# --- Course environment (provided by the lecturers) ---
unique_sdk.api_base = "https://gateway.unique.app/public/chat"
unique_sdk.api_key = API_KEY
unique_sdk.app_id = "app_eyugp8qq242fzp1lle4xjqc1"

COMPANY_ID = "346686658780860534"
USER_ID = "399599116591567964"
SCOPE_IDS = ["scope_nnqbp55xfh6s25wx4mdndff0"]   # knowledge base: company annual reports
MODEL = "AZURE_GPT_51_2025_1113"

In [14]:
# Connection check: one plain LLM call
hello = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID,
    user_id=USER_ID,
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello to the ETH MAS class in one short sentence."},
    ],
)
print(hello.choices[0].message.content)

Hello ETH MAS class—wishing you all an inspiring and successful journey in your studies!


## The question

In [4]:
QUESTION = "Give me an overview of Siemens AG."

## ① Search — what comes back?
The Search API returns **chunks**: text passages with document and page. Nothing has been generated yet.

In [15]:
def search(query: str, limit: int = 5) -> list:
    """Search the knowledge base and return the list of matching chunks."""
    result = unique_sdk.Search.create(
        user_id=USER_ID,
        company_id=COMPANY_ID,
        searchString=query,
        searchType="VECTOR",
        scopeIds=SCOPE_IDS,
        limit=limit,
    )
    return result.data


chunks = search(QUESTION, limit=5)
print(f"{len(chunks)} chunks found\n")
for i, c in enumerate(chunks, 1):
    body = re.sub(r"<\|.*?\|>.*?<\|/.*?\|>", "", c["text"]).strip()
    print(f"[{i}] {c.get('key')}  p. {c.get('startPage')}–{c.get('endPage')}")
    print("    " + body[:180].replace("\n", " ") + " …\n")

5 chunks found

[1] Siemens_2022.pdf  p. 5–5
    # 1. Organization of the Siemens Group and basis of presentation  Siemens is a technology group that is active in nearly all countries of the world, focusing on the areas of automa …

[2] Siemens_2021.pdf  p. 5–5
    <!-- PageBreak -->   Combined Management Report   ## 1. Organization of the Siemens Group and basis of presentation  Siemens is a technology group that is active in nearly all coun …

[3] Siemens_2023.pdf  p. 5–5
    # 1. Organization of the Siemens Group and basis of presentation  Siemens is a technology group that is active in nearly all countries of the world, focusing on the areas of automa …

[4] Siemens_2023.pdf  p. 35–35
    ## 8.5 Significant characteristics of the internal control and risk management system   # 9. Siemens AG  The Annual Financial Statements of Siemens AG have been prepared in accorda …

[5] Siemens_2022.pdf  p. 35–35
    ## 8.5 Significant characteristics of the internal control and risk management 

In [16]:
# The raw JSON of ONE chunk (text shortened)
raw = dict(chunks[0]); raw["text"] = raw["text"][:300] + " …"
print(json.dumps(raw, indent=2, ensure_ascii=False))

{
  "object": "search.search",
  "id": "cont_c5b1bnpxck5drrmgw6rwu8k8",
  "chunkId": "chunk_irogqulh0e4kk6rkjhcswzyc",
  "text": "<|document|>Siemens 2022.pdf<|/document|>\n<|title|>Siemens 2022.pdf - Siemens Report FOR FISCAL 2022<|/title|>\n<|info|>Siemens Report FOR FISCAL 2022 SIEMENS !-- PageBreak -- Table of contents Combined Management Report Consolidated Financial Statements Responsibility Statement Siemens Group Indepen …",
  "createdAt": "2026-11-14T19:41:31.022Z",
  "updatedAt": "2026-11-14T19:47:21.021Z",
  "url": null,
  "key": "Siemens_2022.pdf",
  "title": "Siemens_2022.pdf",
  "internallyStoredAt": "2026-11-14T19:44:47.819Z",
  "order": 2,
  "startPage": 5,
  "endPage": 5,
  "metadata": {
    "key": "Siemens_2022.pdf",
    "title": "Siemens_2022.pdf",
    "folderId": "scope_nnqbp55xfh6s25wx4mdndff0",
    "mimeType": "application/pdf",
    "companyId": "346686658780860534",
    "contentId": "cont_c5b1bnpxck5drrmgw6rwu8k8",
    "validAsOf": "2026-11-14T19:41:31.013Z",
   

## ② Build the prompt — exactly what the LLM will see
The chunks are pasted into the prompt, numbered so the model can cite them.

In [17]:
def build_context(chunks: list) -> str:
    parts = []
    for i, c in enumerate(chunks, 1):
        source = f"{c.get('title') or c.get('key')}, p. {c.get('startPage')}–{c.get('endPage')}"
        parts.append(f"[{i}] ({source})\n{c['text']}")
    return "\n\n".join(parts)

SYSTEM_PROMPT = """You are an analyst. Answer the question using ONLY the context below.
Cite the sources you use as [1], [2], ...
If the context does not contain the answer, say "I could not find this in the knowledge base."
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Context:\n{build_context(chunks)}\n\nQuestion: {QUESTION}"},
]
user_text = messages[1]["content"]
print("SYSTEM:\n" + messages[0]["content"])
print(f"USER ({len(user_text):,} characters):\n{user_text[:900]}\n   [...]\n{user_text[-200:]}")

SYSTEM:
You are an analyst. Answer the question using ONLY the context below.
Cite the sources you use as [1], [2], ...
If the context does not contain the answer, say "I could not find this in the knowledge base."

USER (10,103 characters):
Context:
[1] (Siemens_2022.pdf, p. 5–5)
<|document|>Siemens 2022.pdf<|/document|>
<|title|>Siemens 2022.pdf - Siemens Report FOR FISCAL 2022<|/title|>
<|info|>Siemens Report FOR FISCAL 2022 SIEMENS !-- PageBreak -- Table of contents Combined Management Report Consolidated Financial Statements Responsibility Statement Siemens Group Independent Auditor's Reports Siemens Group Annual Financial Statements Responsibility Statement Siemens AG<|/info|>
# 1.
Organization of the Siemens Group and basis of presentation

Siemens is a technology group that is active in nearly all countries of the world, focusing on the areas of automation and digitalization in
the process and manufacturing industries, intelligent infrastructure for buildings and distributed en

## ③ Generate — one LLM call, the final answer

In [18]:
response = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID, user_id=USER_ID, model=MODEL, messages=messages, options={"temperature": 0.0},
)
print(response.choices[0].message.content)

Siemens AG is the parent company of the Siemens Group, a global technology group headquartered in Munich, Germany. It is organized as a stock corporation (Aktiengesellschaft) under German federal law and prepares its annual financial statements in accordance with the German Commercial Code (Handelsgesetzbuch) and the German Stock Corporation Act (Aktiengesetz). [1][2][3][5]

**Business focus of the Siemens Group (which Siemens AG heads)**  
Siemens operates in nearly all countries worldwide, concentrating on:  
- Automation and digitalization in process and manufacturing industries  
- Intelligent infrastructure for buildings and distributed energy systems  
- Smart mobility solutions for rail transport (and rail and road in 2021)  
- Medical technology and digital healthcare services  

These activities are organized into reportable segments: Digital Industries, Smart Infrastructure, Mobility, Siemens Healthineers (together forming the “Industrial Business”), and Siemens Financial Ser

---
# Solutions to "Your turn"
For the rest we use the three steps in one function.

In [19]:
def rag_answer(question: str, limit: int = 5) -> str:
    chunks = search(question, limit=limit)
    response = unique_sdk.ChatCompletion.create(
        company_id=COMPANY_ID, user_id=USER_ID, model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{build_context(chunks)}\n\nQuestion: {question}"},
        ],
        options={"temperature": 0.0},
    )
    return response.choices[0].message.content

### 1 · With vs. without context
*Expected:* without context the model answers from training knowledge — fluent, but no sources and possibly outdated. With context: grounded and cited, but limited to what was retrieved.

In [20]:
no_context = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID, user_id=USER_ID, model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
)
print("WITHOUT CONTEXT:\n" + no_context.choices[0].message.content[:1200])

WITHOUT CONTEXT:
Siemens AG is a German multinational technology company and one of Europe’s largest industrial groups. Founded in 1847 by Werner von Siemens and Johann Georg Halske, it is headquartered in Munich and operates globally in more than 190 countries.

### Core Business Areas
Siemens focuses on electrification, automation, and digitalization across several main segments:

1. **Digital Industries**
   - Industrial automation hardware (controllers, drives, sensors)
   - Software for product design, manufacturing, and industrial IoT (e.g., Siemens Xcelerator, Teamcenter, NX)
   - Focus: “digital factory” and Industry 4.0 solutions

2. **Smart Infrastructure**
   - Power distribution equipment, building automation, HVAC controls
   - Grid control and smart grid technologies
   - Focus: energy-efficient buildings and resilient urban infrastructure

3. **Mobility**
   - Rail vehicles (trains, metros, trams), signaling and control systems
   - Rail infrastructure, electrification, 

### 2 · How much context? `limit=1` vs `limit=10`
*Expected:* 1 chunk → thin answer; 10 chunks → richer answer, but a longer prompt = more tokens, cost and latency.

In [21]:
for limit in (1, 10):
    print(f"===== limit={limit} =====")
    print(rag_answer(QUESTION, limit=limit)[:1000], "\n")

===== limit=1 =====
Siemens AG is a German stock corporation and the parent company of the Siemens Group, a global technology company headquartered in Munich and active in nearly all countries of the world. The Group focuses on:

- **Automation and digitalization** in process and manufacturing industries  
- **Intelligent infrastructure** for buildings and distributed energy systems  
- **Smart mobility solutions** for rail transport  
- **Medical technology and digital healthcare services** [1]

As of September 30, 2022, Siemens had around **311,000 employees** worldwide. Its operations are organized into several **reportable segments**:

- **Digital Industries**  
- **Smart Infrastructure**  
- **Mobility**  
- **Siemens Healthineers**  

These four together form the company’s **Industrial Business**. In addition, **Siemens Financial Services (SFS)** supports the industrial businesses and conducts its own business with external customers. Siemens also reports results for **Portfolio 

### 3 · Not in the knowledge base
*Expected:* thanks to the grounding instruction the model says it could not find it — instead of inventing a share price.

In [22]:
print(rag_answer("What is today's share price of Nestlé?"))

I could not find this in the knowledge base.


### 4 · A question with a hop
*Expected:* with 5 chunks the result is unstable (often "could not find"); with 20 chunks it finds **Varian** — but there is **no passage from the 2023 report** and no purchase price. One search cannot target "Varian 2023", because the name is only known after reading the first result. ➜ Exercise 2.

In [23]:
HOP_QUESTION = ("In fiscal 2021 Siemens completed a large acquisition. Which company was it, "
                "what did Siemens pay, and how did that business develop in fiscal 2022 and 2023?")

for limit in (5, 20):
    found = search(HOP_QUESTION, limit=limit)
    reports = sorted({c.get("key") for c in found})
    print(f"===== limit={limit} · reports in the chunks: {reports}")
    print("Any chunk from 2023?", any("2023" in (c.get("key") or "") for c in found))
    print(rag_answer(HOP_QUESTION, limit=limit)[:900], "\n")

===== limit=5 · reports in the chunks: ['Siemens_2021.pdf', 'Siemens_2022.pdf']
Any chunk from 2023? False
I could not find this in the knowledge base. 

===== limit=20 · reports in the chunks: ['Siemens_2021.pdf', 'Siemens_2022.pdf']
Any chunk from 2023? False
In fiscal 2021, the large acquisition Siemens completed was Varian Medical Systems, Inc. (“Varian”).

- **Which company?**  
  Varian Medical Systems, Inc., a global leader in cancer care, especially radiation oncology and related digital solutions, acquired by Siemens Healthineers. [1], [6], [16]

- **What did Siemens pay?**  
  The exact total purchase price is not stated in the provided excerpts. The text only notes that the purchase price allocation was (initially) preliminary and later finalized, with goodwill of €8.2 billion and other intangible assets of €6.2 billion after adjustment, but it does not give the total consideration paid for Varian. [3], [6]  
  Therefore, based on the given context, the total purchase price 